In [ ]:
%pip install ultralytics

In [2]:
import os
import shutil
import csv
from ultralytics import YOLO

# --- CONFIGURAZIONE ---
PATH_MODELLO = 'best-bilance.pt'  # Inserisci il percorso del tuo modello
CARTELLA_INPUT = 'images_inference'      # Cartella contenente le immagini da classificare
CARTELLA_OUTPUT = 'risultati_yolo'     # Cartella dove verranno create le sottocartelle
FILE_CSV = 'report_inference.csv'

def main():
    # 1. Caricamento del modello
    model = YOLO(PATH_MODELLO)
    
    # Crea la cartella di output se non esiste
    if not os.path.exists(CARTELLA_OUTPUT):
        os.makedirs(CARTELLA_OUTPUT)

    # Preparazione per il file CSV
    dati_csv = []

    # 2. Iterazione sulle immagini nella cartella di input
    formati_ammessi = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    immagini = [f for f in os.listdir(CARTELLA_INPUT) if f.lower().endswith(formati_ammessi)]

    print(f"Trovate {len(immagini)} immagini. Inizio inferenza...")

    for nome_file in immagini:
        percorso_img = os.path.join(CARTELLA_INPUT, nome_file)
        
        # Esecuzione inferenza
        results = model.predict(source=percorso_img, save=False, verbose=False)
        
        for result in results:
            # Ottieni l'indice della classe con la probabilità più alta
            class_id = result.probs.top1
            # Ottieni il nome della classe corrispondente
            class_name = result.names[class_id]
            # Ottieni la confidenza (opzionale, utile per il CSV)
            confidence = result.probs.top1conf.item()

            # 3. Organizzazione in cartelle
            cartella_destinazione = os.path.join(CARTELLA_OUTPUT, class_name)
            if not os.path.exists(cartella_destinazione):
                os.makedirs(cartella_destinazione)
            
            # Copia l'immagine nella cartella della classe
            shutil.copy(percorso_img, os.path.join(cartella_destinazione, nome_file))

            # Aggiungi i dati alla lista per il CSV
            dati_csv.append([nome_file, class_name, f"{confidence:.4f}"])

    # 4. Scrittura del file CSV
    with open(FILE_CSV, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Filename', 'Classe_Predetta', 'Confidenza'])
        writer.writerows(dati_csv)

    print(f"Processo completato! Immagini smistate in '{CARTELLA_OUTPUT}' e report salvato in '{FILE_CSV}'.")

if __name__ == "__main__":
    main()

Trovate 2910 immagini. Inizio inferenza...
Processo completato! Immagini smistate in 'risultati_yolo' e report salvato in 'report_inference.csv'.
